# Class-Specific Language Models for LLM Attribution

**Goal:** Test whether a separate causal language model, fine-tuned on each candidate class, assigns lower token-level negative log-likelihood (NLL) to held-out passages from that class.

This notebook is designed for ablation studies: change `BASE_MODEL` and the settings in **Cell 2**, then rerun the notebook.

## Experimental safeguards
- Split by `prompt_id` **before** fitting any class-specific model. Prompts in train and validation are disjoint.
- Fine-tune the 18 class-specific models on the training fold only.
- Score validation passages without updating model weights.
- Report **balanced accuracy** as the primary metric, plus macro-F1.
- Lower mean token NLL means a model finds the passage more probable. This is a useful signal, not proof of authorship.
- This is a computationally expensive experiment: each run fine-tunes one model per class, then scores every validation passage under every class model.

**Important:** The notebook uses a single grouped holdout fold for initial screening. For a robust estimate, repeat with multiple group splits after the pipeline works.

## 1. Configuration — change the model here for ablations

In [ ]:
# Example ablations:
# BASE_MODEL = "gpt2"
# BASE_MODEL = "Qwen/Qwen2.5-0.5B"
#
# Start with GPT-2 Small to validate the pipeline. Qwen2.5-0.5B Base is a
# larger, more modern causal LM and will require more GPU memory/time.

BASE_MODEL = "gpt2"

TRAIN_PATH = None  # e.g. "/kaggle/input/name-that-llm/train.csv"; None = search common locations
OUTPUT_DIR = "/kaggle/working/class_lm_attribution"

RANDOM_STATE = 42
N_SPLITS = 5
VALIDATION_FOLD = 0  # uses the first split produced by StratifiedGroupKFold

MAX_LENGTH = 384       # reduce to 256 if memory/time is limited
TRAIN_EPOCHS = 2       # start small; tune only after pipeline works
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
EVAL_BATCH_SIZE = 4
LOGGING_STEPS = 25

INCLUDE_BASE_MODEL_AS_SCORER = True
SAVE_CLASS_MODELS = False
USE_FP16 = True        # enabled only when CUDA is available


## 2. Install dependencies

In [ ]:
# Run if needed. If packages are upgraded in an existing notebook session,
# restart the kernel and run from the top.
%pip install -q -U transformers accelerate datasets scikit-learn pandas numpy tqdm

## 3. Imports and reproducibility

In [ ]:
import os
import gc
import json
import math
import random
import glob
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score, f1_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)

set_seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
else:
    print("WARNING: No CUDA GPU detected. Training 18 models on CPU may be very slow.")


## 4. Load and validate the data

In [ ]:
def find_train_csv():
    candidates = [
        "/kaggle/input/name-that-llm/train.csv",
        "/kaggle/working/train.csv",
        "./train.csv",
        "../input/train.csv",
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    matches = glob.glob("/kaggle/input/**/train.csv", recursive=True)
    if matches:
        print("Found candidate files:", matches[:10])
        return matches[0]
    raise FileNotFoundError(
        "Could not find train.csv. Set TRAIN_PATH in Cell 2 to the correct path."
    )

csv_path = TRAIN_PATH or find_train_csv()
df = pd.read_csv(csv_path)
print("Loaded:", csv_path, "| shape:", df.shape)
display(df.head())

required = {"text", "model", "prompt_id"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df.dropna(subset=["model", "prompt_id"]).copy()
df["text"] = df["text"].fillna("").astype(str)
df = df[df["text"].str.strip().ne("")].reset_index(drop=True)

print("Rows:", len(df))
print("Classes:", df["model"].nunique())
print("Prompts:", df["prompt_id"].nunique())
display(df["model"].value_counts().rename("passages").to_frame())


## 5. Create a prompt-disjoint train/validation split

In [ ]:
label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df["model"])
class_names = list(label_encoder.classes_)
n_classes = len(class_names)

splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE
)
splits = list(splitter.split(df["text"], df["label_id"], groups=df["prompt_id"]))
train_idx, val_idx = splits[VALIDATION_FOLD]

train_df = df.iloc[train_idx].reset_index(drop=True).copy()
val_df = df.iloc[val_idx].reset_index(drop=True).copy()

train_prompts = set(train_df["prompt_id"])
val_prompts = set(val_df["prompt_id"])
assert train_prompts.isdisjoint(val_prompts), "Leakage: prompt groups overlap!"

print(f"Train rows: {len(train_df)} | validation rows: {len(val_df)}")
print(f"Train prompts: {len(train_prompts)} | validation prompts: {len(val_prompts)}")
print(f"Number of classes: {n_classes}")
print("\nTraining examples per class:")
display(train_df["model"].value_counts().reindex(class_names).rename("n_train").to_frame())
print("\nValidation examples per class:")
display(val_df["model"].value_counts().reindex(class_names).rename("n_val").to_frame())


## 6. Load tokenizer and prepare causal-LM training data

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

# Add a dedicated padding token if the tokenizer has none (e.g., GPT-2).
# This avoids reusing EOS as padding and accidentally masking genuine EOS labels.
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})

print("Model:", BASE_MODEL)
print("Tokenizer:", tokenizer.__class__.__name__)
print("Vocab size:", len(tokenizer))
print("Pad token:", repr(tokenizer.pad_token), "| EOS:", repr(tokenizer.eos_token))


In [ ]:
from datasets import Dataset

def make_lm_dataset(texts, tokenizer, max_length):
    ds = Dataset.from_dict({"text": list(texts)})
    def tokenize_batch(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=max_length,
            padding=False,
        )
    ds = ds.map(tokenize_batch, batched=True, remove_columns=["text"])
    # Drop passages that produce too few tokens for a meaningful causal-LM loss.
    ds = ds.filter(lambda row: len(row["input_ids"]) >= 3)
    return ds

# Training data is created from the training fold only.
class_train_datasets = {}
for class_name in class_names:
    texts = train_df.loc[train_df["model"] == class_name, "text"].tolist()
    class_train_datasets[class_name] = make_lm_dataset(texts, tokenizer, MAX_LENGTH)

print("Tokenized training examples per class:")
display(pd.DataFrame({
    "class": class_names,
    "n_passages": [len(class_train_datasets[c]) for c in class_names]
}))


## 7. Fine-tune one language model per class

Each model starts from the same pretrained checkpoint and is fine-tuned only on training-fold passages for its class. We intentionally do not use validation passages for training or early stopping.

For the first experiment, use a small number of epochs. With only around 150 passages per class, longer fine-tuning can overfit quickly.

In [ ]:
def safe_model_name(name):
    return "".join(ch if ch.isalnum() or ch in "-_." else "_" for ch in name)

def train_class_lm(class_name, train_dataset):
    class_dir = Path(OUTPUT_DIR) / safe_model_name(class_name)
    if SAVE_CLASS_MODELS:
        class_dir.mkdir(parents=True, exist_ok=True)

    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
    # Resize only when tokenizer gained a new padding token.
    if model.get_input_embeddings().num_embeddings != len(tokenizer):
        model.resize_token_embeddings(len(tokenizer))
    model.config.pad_token_id = tokenizer.pad_token_id

    collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    args = TrainingArguments(
        output_dir=str(class_dir if SAVE_CLASS_MODELS else Path(OUTPUT_DIR) / "tmp" / safe_model_name(class_name)),
        num_train_epochs=TRAIN_EPOCHS,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        logging_steps=LOGGING_STEPS,
        logging_strategy="steps",
        save_strategy="no",
        report_to="none",
        fp16=bool(USE_FP16 and torch.cuda.is_available()),
        bf16=False,
        optim="adamw_torch",
        seed=RANDOM_STATE,
        data_seed=RANDOM_STATE,
        dataloader_pin_memory=torch.cuda.is_available(),
        remove_unused_columns=True,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        data_collator=collator,
        processing_class=tokenizer,
    )
    trainer.train()

    if SAVE_CLASS_MODELS:
        trainer.save_model(str(class_dir))
        tokenizer.save_pretrained(str(class_dir))

    # Keep one trained model in memory only long enough to score validation.
    trained_model = trainer.model
    trained_model.eval()
    return trained_model, trainer.state.log_history


## 8. Score passages using mean token negative log-likelihood

In [ ]:
@torch.no_grad()
def score_texts_nll(model, texts, tokenizer, max_length=384, batch_size=4):
    """Return mean next-token NLL per passage (lower = more likely under model).

    Scores are calculated over tokens 2..T, because causal LM logits at position
    t predict token t+1. Padding and the first token are excluded.
    Texts are truncated to max_length for this first experiment.
    """
    model.eval()
    device = next(model.parameters()).device
    scores = []

    for start in tqdm(range(0, len(texts), batch_size), leave=False):
        batch_texts = list(texts[start:start + batch_size])
        enc = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length,
        )
        input_ids = enc["input_ids"].to(device)
        attention_mask = enc["attention_mask"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits[:, :-1, :].contiguous()
        labels = input_ids[:, 1:].contiguous()
        valid = attention_mask[:, 1:].contiguous().bool()

        # Token NLL; ignore padding by selecting only valid shifted labels.
        token_nll = torch.nn.functional.cross_entropy(
            logits.float().view(-1, logits.size(-1)),
            labels.view(-1),
            reduction="none",
        ).view(labels.shape)

        for i in range(token_nll.size(0)):
            mask = valid[i]
            if mask.any():
                scores.append(token_nll[i][mask].mean().item())
            else:
                scores.append(float("inf"))

        del outputs, logits, labels, token_nll, input_ids, attention_mask, enc
    return np.asarray(scores, dtype=np.float64)


## 9. Fit all class-specific models and score the validation fold

The resulting matrix has one row per validation passage and one column per candidate class. The predicted class is the class model with the **lowest mean token NLL**.

This can take a while: each of the 18 models scores every validation passage. If memory is limited, the notebook processes models sequentially.

In [ ]:
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
score_columns = {}
training_logs = {}

val_texts = val_df["text"].tolist()

# Optional reference scorer: the unadapted pretrained model.
if INCLUDE_BASE_MODEL_AS_SCORER:
    print(f"Scoring with unadapted base model: {BASE_MODEL}")
    base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL).to(DEVICE)
    if base_model.get_input_embeddings().num_embeddings != len(tokenizer):
        base_model.resize_token_embeddings(len(tokenizer))
    if tokenizer.pad_token_id is not None:
        base_model.config.pad_token_id = tokenizer.pad_token_id
    score_columns["BASE_PRETRAINED"] = score_texts_nll(
        base_model, val_texts, tokenizer, MAX_LENGTH, EVAL_BATCH_SIZE
    )
    del base_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

for class_name in tqdm(class_names, desc="Class-specific LMs"):
    print(f"\nTraining scorer for: {class_name}")
    model, log_history = train_class_lm(class_name, class_train_datasets[class_name])
    training_logs[class_name] = log_history

    print(f"Scoring validation passages with: {class_name}")
    score_columns[class_name] = score_texts_nll(
        model, val_texts, tokenizer, MAX_LENGTH, EVAL_BATCH_SIZE
    )

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

score_df = pd.DataFrame(score_columns)
score_df.to_csv(Path(OUTPUT_DIR) / "validation_nll_scores.csv", index=False)
print("Saved score matrix:", Path(OUTPUT_DIR) / "validation_nll_scores.csv")
display(score_df.head())


## 10. Evaluate attribution performance

In [ ]:
# Restrict prediction to the 18 class-specific scorers; the shared base model is
# a reference diagnostic and is not a target class.
class_score_matrix = score_df[class_names].to_numpy()
pred_ids = class_score_matrix.argmin(axis=1)
true_ids = val_df["label_id"].to_numpy()

balanced_acc = balanced_accuracy_score(true_ids, pred_ids)
macro_f1 = f1_score(true_ids, pred_ids, average="macro", zero_division=0)

print(f"BASE_MODEL: {BASE_MODEL}")
print(f"Balanced accuracy: {balanced_acc:.5f}")
print(f"Macro-F1:          {macro_f1:.5f}")
print("\nClassification report:")
print(classification_report(
    true_ids, pred_ids,
    labels=np.arange(n_classes),
    target_names=class_names,
    zero_division=0,
    digits=4,
))


## 11. Inspect per-class likelihoods and errors

In [ ]:
predictions = val_df[["id", "prompt_id", "text", "model", "label_id"]].copy() if "id" in val_df.columns else val_df[["prompt_id", "text", "model", "label_id"]].copy()
predictions["predicted_model"] = [class_names[i] for i in pred_ids]
predictions["correct"] = predictions["label_id"].to_numpy() == pred_ids
predictions["best_nll"] = class_score_matrix.min(axis=1)
predictions["second_best_nll"] = np.sort(class_score_matrix, axis=1)[:, 1]
predictions["score_margin"] = predictions["second_best_nll"] - predictions["best_nll"]

display(predictions.head(20))
print("Accuracy by true class:")
per_class = []
for i, name in enumerate(class_names):
    mask = true_ids == i
    per_class.append({
        "class": name,
        "n": int(mask.sum()),
        "recall": float((pred_ids[mask] == i).mean()) if mask.any() else np.nan,
        "mean_true_class_nll": float(class_score_matrix[mask, i].mean()) if mask.any() else np.nan,
    })
per_class_df = pd.DataFrame(per_class).sort_values("recall")
display(per_class_df)


## 12. Save reproducible experiment artifacts

In [ ]:
metadata = {
    "base_model": BASE_MODEL,
    "random_state": RANDOM_STATE,
    "n_splits": N_SPLITS,
    "validation_fold": VALIDATION_FOLD,
    "max_length": MAX_LENGTH,
    "train_epochs": TRAIN_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "include_base_model_as_scorer": INCLUDE_BASE_MODEL_AS_SCORER,
    "balanced_accuracy": float(balanced_acc),
    "macro_f1": float(macro_f1),
    "class_names": class_names,
    "train_rows": int(len(train_df)),
    "validation_rows": int(len(val_df)),
    "train_prompt_count": int(len(train_prompts)),
    "validation_prompt_count": int(len(val_prompts)),
}
with open(Path(OUTPUT_DIR) / "experiment_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

predictions.to_csv(Path(OUTPUT_DIR) / "validation_predictions.csv", index=False)
per_class_df.to_csv(Path(OUTPUT_DIR) / "per_class_metrics.csv", index=False)

print("Artifacts saved under:", OUTPUT_DIR)
print(" - validation_nll_scores.csv")
print(" - validation_predictions.csv")
print(" - per_class_metrics.csv")
print(" - experiment_metadata.json")


## 13. Ablation protocol

For each run, change only `BASE_MODEL` first and rerun the notebook from the top.

Suggested sequence:
1. `gpt2`
2. `Qwen/Qwen2.5-0.5B` (use the **Base** checkpoint, not an instruction-tuned chat checkpoint)

Keep the split, `MAX_LENGTH`, number of epochs, and scoring procedure fixed for a fair first comparison. Save each run's output directory or rename it before the next run if you want to preserve the score files.

### How to interpret results
- Compare balanced accuracy first, then macro-F1 and per-class recall.
- Compare against the existing grouped-CV TF-IDF baseline (~0.359 balanced accuracy from the earlier experiment).
- If the score is weak, do not immediately increase epochs. First inspect per-class recall and score margins; class-specific models may overfit or simply not learn distinctive patterns from 150 passages.
- This first notebook truncates long passages at `MAX_LENGTH`. If truncation is substantial, implement sliding-window scoring in a follow-up experiment.
- For final conclusions, repeat with multiple prompt-grouped folds. A single fold is only a screening result.